# Python reference 3: lists, tuples, dicts and sets

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-reference-03-lists-dicts-sets.ipynb)

List methods and slicing, tuples and unpacking, dict reading, merging and views, set operations, comprehensions, sorting with keys and stability, copying and aliasing, and the complexity of common operations. Practise it in [Python practice 3: lists, tuples, dicts and sets](../practice/python-practice-03-lists-dicts-sets.ipynb).

*Last reviewed: November 2026, against Python 3.13.*

## Contents

| section | entries |
| --- | --- |
| [Lists](#lists) | [List methods](#list-methods), [Slicing and slice assignment](#slicing-and-slice-assignment) |
| [Tuples and unpacking](#tuples-and-unpacking) | [Tuples, packing and unpacking](#tuples-packing-and-unpacking) |
| [Dictionaries](#dictionaries) | [Creating and reading dicts](#creating-and-reading-dicts), [Updating, merging and removing](#updating-merging-and-removing), [Views: `keys()`, `values()`, `items()`](#views-keys-values-items) |
| [Sets](#sets) | [Set operations](#set-operations) |
| [Comprehensions](#comprehensions) | [List, dict, set and generator comprehensions](#list-dict-set-and-generator-comprehensions) |
| [Sorting](#sorting) | [`sorted()`, `list.sort()`, `key` and stability](#sorted-listsort-key-and-stability) |
| [Copying, aliasing and mutability](#copying-aliasing-and-mutability) | [Copying, aliasing and mutability](#copying-aliasing-and-mutability) |
| [Performance](#performance) | [Time complexity of common operations](#time-complexity-of-common-operations) |

## Lists

### List methods

A `list` is an ordered, mutable sequence. Methods that change the list in place return `None`.

| method | effect | returns | cost |
| --- | --- | --- | --- |
| `append(x)` | add one item at the end | `None` | O(1) |
| `extend(iterable)` / `+=` | add many items | `None` | O(k) |
| `insert(i, x)` | insert before index `i` | `None` | O(n) |
| `remove(x)` | delete the first item equal to `x` (`ValueError` if absent) | `None` | O(n) |
| `pop(i=-1)` | remove and return an item (`IndexError` if empty) | the item | O(1) at the end, O(n) elsewhere |
| `clear()` | remove everything | `None` | O(n) |
| `index(x, start, stop)` | position of the first `x` (`ValueError` if absent) | int | O(n) |
| `count(x)` | occurrences | int | O(n) |
| `sort(key=None, reverse=False)` | sort in place, stable | `None` | O(n log n) |
| `reverse()` | reverse in place | `None` | O(n) |
| `copy()` | shallow copy | new list | O(n) |

**Notes:** `x = x.sort()` sets `x` to `None`; use `sorted(x)` for a new list. Use `collections.deque` when you pop from
the front.

In [ ]:
items = [3, 1, 2]
items.append(4); items.extend([5, 6]); items.insert(0, 0); items.remove(3)
print("after append/extend/insert/remove:", items)
last, first = items.pop(), items.pop(0)
print("popped", last, "and", first, "->", items, "| index(2):", items.index(2), "| count(1):", items.count(1))
result = items.sort()
print("sort() returns:", result, "| sorted copy:", sorted([3, 1, 2], reverse=True))

### Slicing and slice assignment

`seq[start:stop:step]` returns a new list; omitted values default to the ends. Slices never raise `IndexError`. Assigning
to a slice replaces that part in place, and can change the length.

In [ ]:
nums = list(range(10))
print(nums[2:5], nums[:3], nums[-3:], nums[::2], nums[::-1], nums[8:100])
nums[2:5] = ["a"]
print(nums)
nums[:] = []  # clear in place, visible to every alias
print(nums)

## Tuples and unpacking

### Tuples, packing and unpacking

A `tuple` is an immutable sequence: a fixed-shape record. Tuples of hashable values are hashable, so they work as dict
keys and set members (composite keys).

| syntax | meaning |
| --- | --- |
| `t = 1, 2` / `(1,)` | packing; a one-item tuple needs the comma |
| `a, b = b, a` | swap |
| `first, *rest = seq` / `*init, last = seq` | star unpacking into a list |
| `for key, value in pairs:` | unpack in loops |
| `{(region, day): total}` | composite dictionary key |

**Notes:** a tuple holding a list is still mutable inside: `t = ([],)`; `t[0].append(1)` works, but `hash(t)` fails.

In [ ]:
a, b = 1, 2
a, b = b, a
first, *middle, last = [1, 2, 3, 4, 5]
print(a, b, first, middle, last, (1,), type((1)))
totals = {("north", "2026-01-05"): 120, ("south", "2026-01-05"): 80}
print(totals[("north", "2026-01-05")])
try:
    hash(([],))
except TypeError as error:
    print("TypeError:", error)

## Dictionaries

### Creating and reading dicts

Dicts map hashable keys to values and keep **insertion order** (guaranteed since 3.7).

| operation | behaviour |
| --- | --- |
| `d[key]` | value, or `KeyError` |
| `d.get(key, default=None)` | value or default, never raises |
| `d.setdefault(key, default)` | value; inserts `default` first if the key is missing |
| `key in d` | membership on keys, O(1) |
| `dict(a=1)`, `dict(pairs)`, `{k: v for ...}` | construction |
| `dict.fromkeys(keys, value)` | every key gets the **same** `value` object (dangerous with a mutable value) |

In [ ]:
config = {"host": "db", "port": 5432}
print(config["host"], config.get("timeout"), config.get("timeout", 30), "port" in config)
groups = {}
for word in ["apple", "avocado", "bean"]:
    groups.setdefault(word[0], []).append(word)
print(groups)
shared = dict.fromkeys(["a", "b"], [])
shared["a"].append(1)
print("fromkeys trap:", shared)

### Updating, merging and removing

| operation | effect |
| --- | --- |
| `d[key] = value` | insert or replace |
| `d.update(other_or_pairs)` | merge in place, later wins |
| `d1 \| d2` / `d1 \|= d2` (3.9+) | new merged dict / merge in place, right side wins |
| `{**d1, **d2}` | merged copy (older syntax) |
| `d.pop(key, default)` | remove and return; `KeyError` without a default |
| `d.popitem()` | remove the last inserted item |
| `del d[key]` | remove |

**Notes:** changing a dict's size while iterating over it raises `RuntimeError`; iterate over `list(d)` or build a new
dict instead.

In [ ]:
defaults = {"retries": 3, "timeout": 30}
overrides = {"timeout": 60, "debug": True}
print(defaults | overrides, {**defaults, **overrides})
cfg = dict(defaults)
print(cfg.pop("retries"), cfg.pop("missing", None), cfg)
data = {"a": 1, "b": 0, "c": 2}
try:
    for key in data:
        if data[key] == 0:
            del data[key]
except RuntimeError as error:
    print("RuntimeError:", error)
print({k: v for k, v in data.items() if v != 0})

### Views: `keys()`, `values()`, `items()`

These return live **views** that reflect later changes. `keys()` and `items()` support set operations (`&`, `|`, `-`),
handy for comparing two dicts.

In [ ]:
old = {"a": 1, "b": 2, "c": 3}
new = {"b": 2, "c": 30, "d": 4}
print("added:", new.keys() - old.keys(), "| removed:", old.keys() - new.keys(),
      "| changed:", {k for k in old.keys() & new.keys() if old[k] != new[k]})
keys = old.keys()
old["z"] = 0
print("view sees new key:", "z" in keys)

## Sets

### Set operations

A `set` holds unique hashable items with O(1) membership. `frozenset` is the immutable (hashable) version.

| operation | operator | method |
| --- | --- | --- |
| union | `a \| b` | `a.union(*others)` |
| intersection | `a & b` | `a.intersection(...)` |
| difference | `a - b` | `a.difference(...)` |
| symmetric difference | `a ^ b` | `a.symmetric_difference(b)` |
| subset / superset | `a <= b`, `a >= b` | `issubset`, `issuperset`, `isdisjoint` |
| add / remove | | `add(x)`, `remove(x)` (`KeyError` if absent), `discard(x)` (no error), `pop()` |

**Notes:** sets are unordered; to deduplicate while **keeping order**, use `list(dict.fromkeys(items))`. Methods accept
any iterable; operators need sets.

In [ ]:
a, b2 = {1, 2, 3}, {2, 3, 4}
print(a | b2, a & b2, a - b2, a ^ b2, {2} <= a, a.isdisjoint({9}))
seen = {"x"}
seen.discard("missing"); seen.add("y")
print(sorted(seen), list(dict.fromkeys([3, 1, 3, 2, 1])), a.union([7, 8]))

## Comprehensions

### List, dict, set and generator comprehensions

| form | builds |
| --- | --- |
| `[expr for x in it if cond]` | list |
| `{k: v for x in it}` | dict |
| `{expr for x in it}` | set |
| `(expr for x in it)` | lazy generator ([iterators](python-04-iterators-and-generators.ipynb)) |
| `[a for row in grid for a in row]` | nested loops, outer loop first |
| `[y for x in it if (y := f(x)) > 0]` | the walrus operator reuses a computed value |

**Notes:** `if` at the end filters; a conditional **expression** before `for` maps: `[x if x > 0 else 0 for x in it]`.
Keep comprehensions to one or two clauses; use a loop when they stop being readable.

In [ ]:
rows = [{"sku": "A", "qty": 2}, {"sku": "B", "qty": 0}, {"sku": "C", "qty": 5}]
print([r["sku"] for r in rows if r["qty"] > 0], {r["sku"]: r["qty"] for r in rows}, {r["qty"] > 0 for r in rows})
grid = [[1, 2], [3, 4]]
print([v for row in grid for v in row], [v if v % 2 else 0 for v in range(5)])
print([y for x in ["1", "x", "3"] if (y := int(x) if x.isdigit() else None) is not None])

## Sorting

### `sorted()`, `list.sort()`, `key` and stability

`sorted(iterable, key=None, reverse=False)` returns a new list; `list.sort` sorts in place. Both are **stable**: items
with equal keys keep their original order, which allows multi-pass sorting.

| need | technique |
| --- | --- |
| by a field | `key=lambda r: r["salary"]` or `key=itemgetter("salary")` |
| by an attribute | `key=attrgetter("name")` |
| several fields, all ascending | `key=lambda r: (r["dept"], r["name"])` |
| mixed directions, numeric field | negate it: `key=lambda r: (r["dept"], -r["salary"])` |
| mixed directions, any type | sort by the secondary key first, then by the primary (stability) |
| `None` values | `key=lambda r: (r["x"] is None, r["x"])` puts `None` last |
| case-insensitive | `key=str.casefold` |

In [ ]:
from operator import attrgetter, itemgetter

staff = [{"name": "cai", "dept": "eng", "salary": 90}, {"name": "ana", "dept": "data", "salary": 100},
         {"name": "ben", "dept": "eng", "salary": 120}, {"name": "dee", "dept": "data", "salary": 100}]
print([s["name"] for s in sorted(staff, key=itemgetter("dept", "name"))])
print([s["name"] for s in sorted(staff, key=lambda s: (s["dept"], -s["salary"]))])
by_name_desc = sorted(staff, key=itemgetter("name"), reverse=True)
print([s["name"] for s in sorted(by_name_desc, key=itemgetter("dept"))], "(dept asc, then name desc via two stable sorts)")
print(sorted([3, None, 1], key=lambda x: (x is None, x)), sorted(["b", "A", "c"], key=str.casefold))

## Copying, aliasing and mutability

### Copying, aliasing and mutability

| operation | result |
| --- | --- |
| `b = a` | the **same** object (an alias) |
| `a.copy()`, `list(a)`, `a[:]`, `dict(d)`, `copy.copy(x)` | a **shallow** copy: new outer container, same inner objects |
| `copy.deepcopy(x)` | copies nested objects too |
| `[[0] * 3] * 3` | three references to **one** inner list |
| `[[0] * 3 for _ in range(3)]` | three independent rows |

**Notes:** functions receive references, so mutating an argument changes the caller's object. Mutable default
arguments are created once ([functions](python-03-control-flow-and-functions.ipynb)).

In [ ]:
import copy

nested = {"tags": ["a"], "n": 1}
shallow, deep = copy.copy(nested), copy.deepcopy(nested)
nested["tags"].append("b")
print(shallow["tags"], deep["tags"])
bad = [[0] * 3] * 3
good = [[0] * 3 for _ in range(3)]
bad[0][0] = good[0][0] = 1
print(bad, good)

## Performance

### Time complexity of common operations

| operation | list | dict / set |
| --- | --- | --- |
| index / key lookup | O(1) | O(1) average |
| `x in container` | **O(n)** | O(1) average |
| append / add | O(1) amortized | O(1) average |
| insert / delete at the front | O(n) (use `deque`) | — |
| delete by key | O(n) (`remove`) | O(1) |
| sort | O(n log n) | — |
| iterate | O(n) | O(n) |

**Notes:** the most common performance bug in Python data code is `if x in some_list` inside a loop: O(n²). Convert the
list to a set once.

In [ ]:
import timeit

ids = list(range(20_000))
as_set = set(ids)
list_ms = timeit.timeit(lambda: 19_999 in ids, number=200) * 1000
set_ms = timeit.timeit(lambda: 19_999 in as_set, number=200) * 1000
print(f"200 lookups: list {list_ms:.2f} ms | set {set_ms:.3f} ms | about {list_ms / set_ms:,.0f}x faster")

**Next:** practise these entries in [Python practice 3: lists, tuples, dicts and sets](../practice/python-practice-03-lists-dicts-sets.ipynb).